In [8]:
# !pip install transformers torch datasets huggingface_hub pillow opencv-python ipywidgets

In [9]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
import sys
import torch

# GPU checks
print("Python:", sys.executable)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Torch CUDA version:", torch.version.cuda)
print("cuDNN enabled:", torch.backends.cudnn.enabled)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

Python: c:\Users\DELL\anaconda3\envs\DeepFakeDetection\python.exe
Torch version: 2.5.1+cu121
CUDA available: True
Torch CUDA version: 12.1
cuDNN enabled: True
True
NVIDIA GeForce RTX 3060 Laptop GPU


# Evaluator (dh batee2, esta5demo el ta7t)

In [ ]:
import torch
from transformers import pipeline, AutoImageProcessor

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

processor = AutoImageProcessor.from_pretrained(
    "buildborderless/CommunityForensics-DeepfakeDet-ViT"
)
processor.size = {"height": 384, "width": 384}

img_pipeline = pipeline(
    "image-classification",
    model="buildborderless/CommunityForensics-DeepfakeDet-ViT",
    image_processor=processor,
    device=0 if torch.cuda.is_available() else -1,
)



In [ ]:
print(img_pipeline.model.config.id2label)
print(img_pipeline.model.config.label2id)

label_map = {       # n8ayaro 3la asas el model, mape el real le 0 wl fake le 1
    "LABEL_0": 0,
    "LABEL_1": 1,
}

In [ ]:
from evaluator import evaluate_all_datasets
results = evaluate_all_datasets(
    pipeline_obj=img_pipeline,
    model_name="CommunityForensics-DeepfakeDet-ViT_TEST",
    batch_size=16,
    checkpoint_every=50,    # saves kol 50 batch
    # max_batches=3,        # uncomment lw hn-test 7aga bs 3la kam batch kda
    label_map=label_map,
)

# Faster Evaluator

In [11]:
import torch
from transformers import AutoImageProcessor, AutoModelForImageClassification

model_name = "buildborderless/CommunityForensics-DeepfakeDet-ViT"

processor = AutoImageProcessor.from_pretrained(model_name)
processor.size = {"height": 384, "width": 384}

model = AutoModelForImageClassification.from_pretrained(model_name)


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

In [12]:

print(model.config.id2label)
print(model.config.label2id)

label_map = {   # n8ayaro 3la asas el model, mape el real le 0 wl fake le 1
    "LABEL_0": 0,  # real
    "LABEL_1": 1,  # fake
}

{0: 'LABEL_0', 1: 'LABEL_1'}
{'LABEL_0': 0, 'LABEL_1': 1}


In [ ]:
from faster_evaluator import evaluate_all_datasets_fast

results = evaluate_all_datasets_fast(
    model=model,
    processor=processor,
    model_name="CommunityForensics-DeepfakeDet-ViT_TEST",
    batch_size=16,
    label_map=label_map,
    checkpoint_every=100,
    num_workers=4,
    # max_batches=3,
)


[1/4] Running dataset: 20K_real_and_deepfake_images
[1/4] Running 20K_real_and_deepfake_images: 17354 remaining


[1/4] 20K_real_and_deepfake_images:   0%|          | 0/543 [00:00<?, ?it/s]

MemoryError: Caught MemoryError in DataLoader worker process 1.
Original Traceback (most recent call last):
  File "c:\Users\DELL\anaconda3\envs\DeepFakeDetection\Lib\site-packages\torch\utils\data\_utils\worker.py", line 351, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\DELL\anaconda3\envs\DeepFakeDetection\Lib\site-packages\torch\utils\data\_utils\fetch.py", line 55, in fetch
    return self.collate_fn(data)
           ^^^^^^^^^^^^^^^^^^^^^
  File "c:\Extra_D\Ali_Master\Term 3\Computer Vision\Project\DeepFakeDetectionAndPrevention_AKA_DEEPFAKEPOLICE\faster_evaluator.py", line 40, in __call__
    inputs = self.processor(images=images, return_tensors="pt")
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\DELL\anaconda3\envs\DeepFakeDetection\Lib\site-packages\transformers\image_processing_utils.py", line 217, in __call__
    return self.preprocess(images, *args, **kwa